# TP voix synthétique, impairments et fixture FlowSynth

Ce TP de 60 minutes fabrique une courte **voix artificielle à formants** avec la seule bibliothèque standard Python, l'encode réellement en G.711 PCMU, la découpe en trames de 20 ms, applique un modèle déterministe de perte et de gigue, puis la reconstruit derrière un jitter buffer. Les deux WAV restent en mémoire et sont écoutables par des URI `data:`.

**Public cible :** étudiant paquet/VoIP, analyste cyber débutant ou novice QoS connaissant IPv4/UDP et les bases de Python. Le livrable attendu est une explication reliant son, trame, séquence, timestamp, perte, arrivée tardive et jitter buffer.

Le dernier volet exporte un scénario `.fs` borné pour Secureworks FlowSynth, mais **n'importe pas et n'exécute jamais FlowSynth**. Il ne crée ni socket, ni PCAP, ni trafic. Les adresses appartiennent aux blocs TEST-NET.

Limites indispensables : le signal n'est **ni une voix humaine ni un TTS**. Chaque payload porte bien un en-tête RTP v2 de 12 octets et du PCMU, mais le notebook ne constitue pas une pile ou une session RTP conforme et n'implémente ni SIP, ni SDP, ni RTCP. Le DSL FlowSynth ne porte aucun timing réseau et un éventuel PCAP compilé ne prouverait donc ni les intervalles de 20 ms, ni la gigue, ni la performance. Les KPI sont ceux de ce modèle pédagogique synthétique, pas des scores perceptuels MOS/PESQ ni une preuve de production.

In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import json
import math
import random
import re
import statistics
import struct
import wave

SAMPLE_RATE_HZ = 8_000
SAMPLE_WIDTH_BYTES = 2
CHANNELS = 1
FRAME_DURATION_MS = 20
SAMPLES_PER_FRAME = SAMPLE_RATE_HZ * FRAME_DURATION_MS // 1_000
SEGMENT_DURATION_S = 0.4
FORMANT_SEGMENTS = (
    ('a', (730.0, 1_090.0, 2_440.0)),
    ('i', (270.0, 2_290.0, 3_010.0)),
    ('u', (300.0, 870.0, 2_240.0)),
    ('e', (530.0, 1_840.0, 2_480.0)),
)
SEED = 20_260_730
LOSS_RATE = 0.07
BASE_DELAY_MS = 45.0
JITTER_LOW_MS = -20.0
JITTER_HIGH_MS = 45.0
PLAYOUT_DELAY_MS = 72.0
FIXED_SSRC = 0x564F4958  # 'VOIX', identifiant RTP pédagogique fixe
RTP_PAYLOAD_TYPE_PCMU = 0
RTP_START_SEQUENCE = 20_000
RTP_START_TIMESTAMP = 1_600_000
FLOWSYNTH_EXECUTED = False

assert SAMPLES_PER_FRAME == 160
print(f'{SAMPLE_RATE_HZ} Hz, G.711 PCMU, {SAMPLES_PER_FRAME} échantillons/20 ms, seed={SEED}')

## 1. Synthèse déterministe et WAV en mémoire

Quatre segments utilisent des fréquences de formants différentes. Cette addition de sinusoïdes et son enveloppe attaque/relâchement donnent seulement une texture *voice-like*. Aucun enregistrement, modèle vocal, texte, identité ou donnée personnelle n'intervient.

In [ ]:
def synthesize_formant_voice() -> tuple[int, ...]:
    segment_samples = round(SEGMENT_DURATION_S * SAMPLE_RATE_HZ)
    samples: list[int] = []
    for segment_index, (_label, formants) in enumerate(FORMANT_SEGMENTS):
        for local_index in range(segment_samples):
            local_t = local_index / SAMPLE_RATE_HZ
            absolute_t = (segment_index * segment_samples + local_index) / SAMPLE_RATE_HZ
            attack = min(1.0, local_t / 0.035)
            release = min(1.0, (SEGMENT_DURATION_S - local_t) / 0.055)
            envelope = max(0.0, min(attack, release))
            f0 = 118.0 + 4.0 * math.sin(2.0 * math.pi * 2.2 * absolute_t)
            glottal = (
                0.58 * math.sin(2.0 * math.pi * f0 * absolute_t)
                + 0.22 * math.sin(2.0 * math.pi * 2.0 * f0 * absolute_t)
                + 0.10 * math.sin(2.0 * math.pi * 3.0 * f0 * absolute_t)
            )
            resonances = (
                0.28 * math.sin(2.0 * math.pi * formants[0] * absolute_t)
                + 0.16 * math.sin(2.0 * math.pi * formants[1] * absolute_t)
                + 0.08 * math.sin(2.0 * math.pi * formants[2] * absolute_t)
            )
            value = round(12_500 * envelope * (0.68 * glottal + 0.32 * resonances))
            samples.append(max(-32_768, min(32_767, value)))
    return tuple(samples)


def pcm16_to_wav(samples: tuple[int, ...]) -> bytes:
    frames = struct.pack(f'<{len(samples)}h', *samples)
    buffer = io.BytesIO()
    with wave.open(buffer, 'wb') as stream:
        stream.setnchannels(CHANNELS)
        stream.setsampwidth(SAMPLE_WIDTH_BYTES)
        stream.setframerate(SAMPLE_RATE_HZ)
        stream.writeframes(frames)
    return buffer.getvalue()


def wav_contract(payload: bytes) -> dict[str, int | str]:
    with wave.open(io.BytesIO(payload), 'rb') as stream:
        contract = {
            'channels': stream.getnchannels(),
            'sample_width_bytes': stream.getsampwidth(),
            'sample_rate_hz': stream.getframerate(),
            'frames': stream.getnframes(),
            'compression': stream.getcomptype(),
        }
    assert payload[:4] == b'RIFF' and payload[8:12] == b'WAVE'
    assert contract == {
        'channels': 1,
        'sample_width_bytes': 2,
        'sample_rate_hz': 8_000,
        'frames': len(original_samples),
        'compression': 'NONE',
    }
    return contract


original_samples = synthesize_formant_voice()
original_wav = pcm16_to_wav(original_samples)
original_sha256 = hashlib.sha256(original_wav).hexdigest()
assert len(original_samples) % SAMPLES_PER_FRAME == 0
wav_contract(original_wav)
print(f'{len(original_samples)} échantillons, {len(original_wav)} octets WAV, sha256={original_sha256}')

## 2. Trames de 20 ms, perte, gigue et jitter buffer

Chaque bloc contient 160 échantillons PCM16 encodés en 160 octets G.711 PCMU, soit 20 ms à 8 kHz. Le réseau pédagogique ajoute un délai aléatoire local, perd certaines trames, puis le récepteur décode et rejoue à échéance fixe. Une trame absente ou arrivée après son échéance est masquée par une répétition atténuée de la dernière bonne trame décodée.

Ce modèle distingue perte réseau, retard excessif, réordonnancement et dissimulation. Il implémente le codage/décodage PCMU et la structure minimale d'un en-tête RTP, mais ni session SIP/SDP/RTCP, ni sockets, ni congestion ; son estimateur de gigue n'est pas une certification RTP.

In [ ]:
PCMU_BIAS = 0x84
PCMU_CLIP = 32_635


def linear16_to_pcmu(sample: int) -> int:
    sign = 0x80 if sample < 0 else 0x00
    magnitude = min(abs(sample), PCMU_CLIP) + PCMU_BIAS
    exponent = max(0, min(7, magnitude.bit_length() - 8))
    mantissa = (magnitude >> (exponent + 3)) & 0x0F
    return (~(sign | (exponent << 4) | mantissa)) & 0xFF


def pcmu_to_linear16(code: int) -> int:
    inverted = (~code) & 0xFF
    sign = inverted & 0x80
    exponent = (inverted >> 4) & 0x07
    mantissa = inverted & 0x0F
    magnitude = ((mantissa << 3) + PCMU_BIAS) << exponent
    sample = magnitude - PCMU_BIAS
    return -sample if sign else sample


def encode_pcmu_frame(frame: tuple[int, ...]) -> bytes:
    assert len(frame) == SAMPLES_PER_FRAME
    return bytes(linear16_to_pcmu(sample) for sample in frame)


def decode_pcmu_frame(frame: bytes) -> tuple[int, ...]:
    assert len(frame) == SAMPLES_PER_FRAME
    return tuple(pcmu_to_linear16(code) for code in frame)


assert (linear16_to_pcmu(0), linear16_to_pcmu(PCMU_CLIP), linear16_to_pcmu(-PCMU_CLIP)) == (0xFF, 0x80, 0x00)
assert (pcmu_to_linear16(0xFF), pcmu_to_linear16(0x80), pcmu_to_linear16(0x00)) == (0, 32_124, -32_124)


def split_frames(samples: tuple[int, ...]) -> tuple[tuple[int, ...], ...]:
    return tuple(
        tuple(samples[start:start + SAMPLES_PER_FRAME])
        for start in range(0, len(samples), SAMPLES_PER_FRAME)
    )


def nearest_rank(values: list[float], quantile: float) -> float:
    ordered = sorted(values)
    if not ordered:
        return 0.0
    rank = max(1, math.ceil(quantile * len(ordered)))
    return ordered[rank - 1]


def impair_and_playout(
    frames: tuple[bytes, ...], seed: int
) -> tuple[tuple[int, ...], dict[str, object], tuple[dict[str, float | int], ...]]:
    rng = random.Random(seed)
    received: dict[int, dict[str, float | int]] = {}
    arrival_order: list[dict[str, float | int]] = []
    network_lost: set[int] = set()
    for sequence, _frame in enumerate(frames):
        sent_at_ms = sequence * FRAME_DURATION_MS
        if rng.random() < LOSS_RATE:
            network_lost.add(sequence)
            continue
        delay_ms = BASE_DELAY_MS + rng.uniform(JITTER_LOW_MS, JITTER_HIGH_MS)
        record = {
            'sequence': sequence,
            'sent_at_ms': float(sent_at_ms),
            'delay_ms': delay_ms,
            'arrival_ms': sent_at_ms + delay_ms,
        }
        received[sequence] = record
        arrival_order.append(record)

    arrival_order.sort(key=lambda item: (float(item['arrival_ms']), int(item['sequence'])))
    max_sequence_seen = -1
    reordered = 0
    for record in arrival_order:
        sequence = int(record['sequence'])
        if sequence < max_sequence_seen:
            reordered += 1
        max_sequence_seen = max(max_sequence_seen, sequence)

    output_frames: list[tuple[int, ...]] = []
    late: set[int] = set()
    last_good = tuple(0 for _ in range(SAMPLES_PER_FRAME))
    for sequence, frame in enumerate(frames):
        deadline_ms = sequence * FRAME_DURATION_MS + PLAYOUT_DELAY_MS
        record = received.get(sequence)
        if record is None or float(record['arrival_ms']) > deadline_ms:
            if record is not None:
                late.add(sequence)
            concealed = tuple(round(sample * 0.72) for sample in last_good)
            output_frames.append(concealed)
        else:
            decoded_frame = decode_pcmu_frame(frame)
            output_frames.append(decoded_frame)
            last_good = decoded_frame

    delivered_by_sequence = [received[index] for index in sorted(received)]
    delays = [float(item['delay_ms']) for item in delivered_by_sequence]
    jitter_estimate = 0.0
    previous_transit: float | None = None
    for delay in delays:
        if previous_transit is not None:
            jitter_estimate += (abs(delay - previous_transit) - jitter_estimate) / 16.0
        previous_transit = delay

    flattened = tuple(sample for frame in output_frames for sample in frame)
    metrics: dict[str, object] = {
        'seed': seed,
        'frames_sent': len(frames),
        'frames_received': len(received),
        'frames_network_lost': len(network_lost),
        'frames_late_at_playout': len(late),
        'frames_concealed': len(network_lost | late),
        'reordered_arrivals': reordered,
        'network_loss_rate': len(network_lost) / len(frames),
        'effective_concealment_rate': len(network_lost | late) / len(frames),
        'mean_delay_ms': statistics.fmean(delays) if delays else 0.0,
        'p95_delay_ms': nearest_rank(delays, 0.95),
        'jitter_estimate_ms': jitter_estimate,
        'playout_delay_ms': PLAYOUT_DELAY_MS,
    }
    return flattened, metrics, tuple(arrival_order)


pcm_frames = split_frames(original_samples)
frames = tuple(encode_pcmu_frame(frame) for frame in pcm_frames)
pcmu_reference_samples = tuple(sample for frame in frames for sample in decode_pcmu_frame(frame))
pcmu_reference_wav = pcm16_to_wav(pcmu_reference_samples)
pcmu_reference_sha256 = hashlib.sha256(pcmu_reference_wav).hexdigest()
degraded_samples, network_metrics, arrival_trace = impair_and_playout(frames, SEED)
degraded_wav = pcm16_to_wav(degraded_samples)
degraded_sha256 = hashlib.sha256(degraded_wav).hexdigest()
wav_contract(degraded_wav)
assert len(frames) == 80 and all(len(frame) == 160 for frame in frames)
assert len(degraded_samples) == len(original_samples)
print(json.dumps(network_metrics, ensure_ascii=False, indent=2, sort_keys=True))

In [ ]:
absolute_error = [abs(a - b) for a, b in zip(original_samples, degraded_samples, strict=True)]
kpis = {
    **network_metrics,
    'frame_duration_ms': FRAME_DURATION_MS,
    'samples_per_frame': SAMPLES_PER_FRAME,
    'pcmu_payload_bytes_per_frame': SAMPLES_PER_FRAME,
    'pcmu_nominal_bitrate_bps': SAMPLE_RATE_HZ * 8,
    'rtp_header_bytes': 12,
    'duration_s': len(original_samples) / SAMPLE_RATE_HZ,
    'mean_absolute_sample_error': statistics.fmean(absolute_error),
    'original_wav_bytes': len(original_wav),
    'degraded_wav_bytes': len(degraded_wav),
    'original_wav_sha256': original_sha256,
    'pcmu_reference_wav_sha256': pcmu_reference_sha256,
    'degraded_wav_sha256': degraded_sha256,
    'synthetic': True,
    'production_evidence': False,
}
assert int(kpis['frames_received']) + int(kpis['frames_network_lost']) == int(kpis['frames_sent'])
assert int(kpis['frames_concealed']) == int(kpis['frames_network_lost']) + int(kpis['frames_late_at_playout'])
assert int(kpis['frames_network_lost']) > 0
assert int(kpis['frames_late_at_playout']) > 0
assert int(kpis['reordered_arrivals']) > 0
assert 0.0 < float(kpis['effective_concealment_rate']) < 1.0
assert original_sha256 != degraded_sha256
print(json.dumps(kpis, ensure_ascii=False, indent=2, sort_keys=True))

## 3. Écoute locale via URI `data:`

Les lecteurs HTML ci-dessous incorporent les WAV en base64. Ils ne contactent aucun serveur, n'écrivent aucun fichier et n'utilisent pas d'autoplay. Une différence audible illustre seulement cette réalisation synthétique et ne remplace pas un protocole d'écoute, un codec étalon ou une métrique perceptuelle.

In [ ]:
def wav_data_uri(payload: bytes) -> str:
    return 'data:audio/wav;base64,' + base64.b64encode(payload).decode('ascii')


class EmbeddedAudioComparison:
    def __init__(self, original: bytes, degraded: bytes) -> None:
        self.original = original
        self.degraded = degraded

    def _repr_html_(self) -> str:
        return (
            '<div style="display:grid;grid-template-columns:1fr 1fr;gap:1rem">'
            '<div><strong>Original synthétique</strong><br>'
            f'<audio controls preload="metadata" src="{wav_data_uri(self.original)}"></audio></div>'
            '<div><strong>Après PCMU, perte, gigue et jitter buffer</strong><br>'
            f'<audio controls preload="metadata" src="{wav_data_uri(self.degraded)}"></audio></div>'
            '</div>'
        )


audio_comparison = EmbeddedAudioComparison(original_wav, degraded_wav)
assert wav_data_uri(original_wav).startswith('data:audio/wav;base64,UklGR')
audio_comparison

## 4. Scénario `.fs` exportable, sans exécution de FlowSynth

La fixture place chaque trame de 160 octets PCMU derrière un en-tête RTP v2 de 12 octets : payload type statique 0, séquence `+1`, timestamp `+160` et SSRC fixe. Le DSL déclare un flux UDP entre `192.0.2.10` et `192.0.2.20`, réservées à la documentation. Cela valide des octets structurés, pas une session RTP complète.

FlowSynth pourrait compiler séparément ces octets en PCAP, mais il n'apporte ni l'horloge de départ toutes les 20 ms, ni SIP/SDP/RTCP, ni le jitter buffer ou les pertes calculées plus haut. Le lien ne fait que télécharger du texte `.fs` ; le notebook n'appelle jamais le compilateur et interdit `filecontent`.

In [ ]:
FLOW_NAME = 'voix_fixture'
SOURCE_IP, DESTINATION_IP = '192.0.2.10', '192.0.2.20'
SOURCE_PORT, DESTINATION_PORT = 40_000, 40_002
SOURCE_MAC, DESTINATION_MAC = '02:00:00:00:00:10', '02:00:00:00:00:20'


def flowsynth_escape(payload: bytes) -> str:
    return ''.join(f'\\x{byte:02x}' for byte in payload)


def build_rtp_packet(sequence_index: int, pcmu_payload: bytes) -> bytes:
    assert len(pcmu_payload) == SAMPLES_PER_FRAME
    sequence_number = RTP_START_SEQUENCE + sequence_index
    timestamp = RTP_START_TIMESTAMP + sequence_index * SAMPLES_PER_FRAME
    header = struct.pack(
        '!BBHII',
        0x80,
        RTP_PAYLOAD_TYPE_PCMU,
        sequence_number,
        timestamp,
        FIXED_SSRC,
    )
    assert len(header) == 12
    return header + pcmu_payload


def build_flowsynth_scenario(source_frames: tuple[bytes, ...]) -> str:
    lines = [
        '# Fixture voix synthetique : octets seulement, aucun timing reseau',
        (f'flow {FLOW_NAME} udp {SOURCE_IP}:{SOURCE_PORT} > {DESTINATION_IP}:{DESTINATION_PORT} '
         f'(src_mac:{SOURCE_MAC}; dst_mac:{DESTINATION_MAC};);'),
    ]
    for sequence_index, pcmu_payload in enumerate(source_frames):
        packet = build_rtp_packet(sequence_index, pcmu_payload)
        lines.append(f'{FLOW_NAME} > (content:"{flowsynth_escape(packet)}";);')
    return '\n'.join(lines) + '\n'


def validate_flowsynth_fixture(source: str, expected_frames: int) -> None:
    assert source.isascii()
    assert len(source.encode('ascii')) < 150_000
    lowered = source.lower()
    for forbidden in ('filecontent', 'include:', 'tcp.initialize', 'exec(', 'system('):
        assert forbidden not in lowered
    lines = source.splitlines()
    declaration = (f'flow {FLOW_NAME} udp {SOURCE_IP}:{SOURCE_PORT} > {DESTINATION_IP}:{DESTINATION_PORT} '
                   f'(src_mac:{SOURCE_MAC}; dst_mac:{DESTINATION_MAC};);')
    assert lines[1] == declaration
    payload_lines = lines[2:]
    assert len(payload_lines) == expected_frames
    pattern = re.compile(rf'^{FLOW_NAME} > \(content:"(?:\\x[0-9a-f]{{2}})+";\);$')
    assert all(pattern.fullmatch(line) for line in payload_lines)
    for sequence_index, line in enumerate(payload_lines):
        packet = bytes(int(value, 16) for value in re.findall(r'\\x([0-9a-f]{2})', line))
        assert len(packet) == 12 + SAMPLES_PER_FRAME
        first, second, sequence_number, timestamp, ssrc = struct.unpack('!BBHII', packet[:12])
        assert first == 0x80 and second == RTP_PAYLOAD_TYPE_PCMU
        assert sequence_number == RTP_START_SEQUENCE + sequence_index
        assert timestamp == RTP_START_TIMESTAMP + sequence_index * SAMPLES_PER_FRAME
        assert ssrc == FIXED_SSRC


flowsynth_scenario = build_flowsynth_scenario(frames)
validate_flowsynth_fixture(flowsynth_scenario, len(frames))
flowsynth_sha256 = hashlib.sha256(flowsynth_scenario.encode('ascii')).hexdigest()
print(f'{len(frames)} payloads UDP déclarés, {len(flowsynth_scenario)} caractères, sha256={flowsynth_sha256}')

In [ ]:
class FlowSynthDownload:
    def __init__(self, source: str) -> None:
        self.source = source

    def _repr_html_(self) -> str:
        encoded = base64.b64encode(self.source.encode('ascii')).decode('ascii')
        return (
            '<a download="tp_voix_flowsynth.fs" '
            f'href="data:text/plain;base64,{encoded}">Télécharger la fixture .fs</a>'
            '<p><small>Texte inerte : FlowSynth n’est pas exécuté et aucun PCAP n’est produit.</small></p>'
        )


flowsynth_download = FlowSynthDownload(flowsynth_scenario)
assert 'data:text/plain;base64,' in flowsynth_download._repr_html_()
flowsynth_download

## 5. Interpréter sans sur-vendre

À retenir :

- une durée de trame et une échéance de playout sont deux horloges différentes ;
- une trame reçue peut être inutilisable parce qu'elle arrive trop tard ;
- un jitter buffer échange de la latence contre moins de pertes tardives ;
- répéter une trame est une dissimulation rudimentaire, pas un codec résilient ;
- FlowSynth décrit ici les octets et leur ordre de compilation, pas les dates de départ ou d'arrivée ;
- les deux WAV, les URI `data:`, les KPI et le `.fs` sont synthétiques et locaux.

Pour passer à un véritable essai VoIP, il faudrait au minimum une pile RTP/SIP ou WebRTC identifiée, un codec et ses paramètres, une horloge et des captures qualifiées, des métriques normalisées, des répétitions, un générateur actif autorisé et une validation indépendante.

In [ ]:
repeat_original_wav = pcm16_to_wav(synthesize_formant_voice())
repeat_degraded_samples, repeat_metrics, _repeat_trace = impair_and_playout(frames, SEED)
repeat_flowsynth = build_flowsynth_scenario(frames)

assert hashlib.sha256(repeat_original_wav).hexdigest() == original_sha256
assert pcm16_to_wav(repeat_degraded_samples) == degraded_wav
assert repeat_metrics == network_metrics
assert repeat_flowsynth == flowsynth_scenario
assert FLOWSYNTH_EXECUTED is False
assert kpis['synthetic'] is True and kpis['production_evidence'] is False

final_summary = {
    'artifact_class': 'synthetic-pedagogical',
    'flow_count': len(frames),
    'flowsynth_executed': FLOWSYNTH_EXECUTED,
    'flowsynth_sha256': flowsynth_sha256,
    'payload_codec': 'G.711 PCMU',
    'rtp_header_bytes': 12,
    'original_wav_sha256': original_sha256,
    'degraded_wav_sha256': degraded_sha256,
    'network_access': False,
    'privileged_actions': False,
    'production_evidence': False,
    'seed': SEED,
}
print('TP_VALIDATED ' + json.dumps(final_summary, ensure_ascii=False, sort_keys=True))